In [9]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

In [2]:
train_dir = "Dataset/train"
val_dir = "Dataset/val"

In [10]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical"
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical"
)

Found 14165 images belonging to 8 classes.
Found 1201 images belonging to 8 classes.


In [11]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

In [12]:
for layer in base_model.layers:
    layer.trainable = False

In [13]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.5)(x)

predictions = layers.Dense(8, activation="softmax")(x)

model = models.Model(inputs=base_model.input, outputs=predictions)

In [14]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [15]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [17]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10
886/886 ━━━━━━━━━━━━━━━━━━━━ 645s 721ms/step - accuracy: 0.9290 - loss: 0.2293 - val_accuracy: 0.9051 - val_loss: 0.3304
Epoch 2/10
886/886 ━━━━━━━━━━━━━━━━━━━━ 809s 913ms/step - accuracy: 0.9603 - loss: 0.1282 - val_accuracy: 0.9151 - val_loss: 0.2957
Epoch 3/10
886/886 ━━━━━━━━━━━━━━━━━━━━ 692s 781ms/step - accuracy: 0.9642 - loss: 0.1172 - val_accuracy: 0.9134 - val_loss: 0.3346
Epoch 4/10
886/886 ━━━━━━━━━━━━━━━━━━━━ 404s 455ms/step - accuracy: 0.9655 - loss: 0.1062 - val_accuracy: 0.9192 - val_loss: 0.3128
Epoch 5/10
886/886 ━━━━━━━━━━━━━━━━━━━━ 438s 495ms/step - accuracy: 0.9694 - loss: 0.0931 - val_accuracy: 0.9251 - val_loss: 0.3227


In [18]:
model.save("waste_mobilenet_model.keras")

In [19]:
_ = model.save("waste_mobilenet_model.keras")

In [21]:
import os
print(os.listdir())

['app.py', 'Dataset', 'mobilenet_training.ipynb', 'predict.py', 'restructure_dataset.py', 'run_prediction.py', 'sample.jpg', 'test_model.py', 'waste_mobilenet_model.keras', 'waste_model']
